In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=1.0) 

In [3]:
class jokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [7]:
def gen_joke(state: jokeState):
    topic = state['topic']
    joke = model.invoke(f"Generate a joke on the topic {topic}").content
    return {"joke": joke}

def gen_explain(state: jokeState):
    explain = model.invoke(f"Explain this joke:  {state['joke']}").content
    return {"explanation": explain}

In [8]:
checkpointer = InMemorySaver()
graph = StateGraph(jokeState)

graph.add_node("gen_joke", gen_joke)
graph.add_node("gen_explain", gen_explain)

graph.add_edge(START, "gen_joke")
graph.add_edge("gen_joke", "gen_explain")
graph.add_edge("gen_explain", END)

workflow = graph.compile(checkpointer=checkpointer)



In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({"topic": "burger"}, config= config1)

{'topic': 'burger',
 'joke': 'Why did the hamburger go to therapy?\n\nBecause it had a lot of *beef* with its past!',
 'explanation': 'Here\'s the explanation of the joke:\n\n**The Pun:**\n\nThe joke relies on a play on words, or a pun, on the word "**beef**."\n\n*   **Literal Meaning:** In the context of food, "beef" refers to the meat of a cow, which is the primary ingredient in a hamburger.\n*   **Figurative Meaning:** In everyday language, "to have beef with someone" means to have a grievance, a complaint, or a lingering conflict with them. It implies a past issue that hasn\'t been resolved and is causing emotional distress.\n\n**How it Connects to the Joke:**\n\n1.  **The Hamburger:** The subject of the joke is a hamburger.\n2.  **"A lot of beef":** The joke uses the figurative meaning of "beef" to suggest the hamburger has a lot of unresolved problems or negative experiences from its past.\n3.  **"With its past":** This phrase directly links the "beef" to the hamburger\'s history

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the hamburger go to therapy?\n\nBecause it had a lot of *beef* with its past!', 'explanation': 'Here\'s the explanation of the joke:\n\n**The Pun:**\n\nThe joke relies on a play on words, or a pun, on the word "**beef**."\n\n*   **Literal Meaning:** In the context of food, "beef" refers to the meat of a cow, which is the primary ingredient in a hamburger.\n*   **Figurative Meaning:** In everyday language, "to have beef with someone" means to have a grievance, a complaint, or a lingering conflict with them. It implies a past issue that hasn\'t been resolved and is causing emotional distress.\n\n**How it Connects to the Joke:**\n\n1.  **The Hamburger:** The subject of the joke is a hamburger.\n2.  **"A lot of beef":** The joke uses the figurative meaning of "beef" to suggest the hamburger has a lot of unresolved problems or negative experiences from its past.\n3.  **"With its past":** This phrase directly links the "beef" to the h

In [12]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the hamburger go to therapy?\n\nBecause it had a lot of *beef* with its past!', 'explanation': 'Here\'s the explanation of the joke:\n\n**The Pun:**\n\nThe joke relies on a play on words, or a pun, on the word "**beef**."\n\n*   **Literal Meaning:** In the context of food, "beef" refers to the meat of a cow, which is the primary ingredient in a hamburger.\n*   **Figurative Meaning:** In everyday language, "to have beef with someone" means to have a grievance, a complaint, or a lingering conflict with them. It implies a past issue that hasn\'t been resolved and is causing emotional distress.\n\n**How it Connects to the Joke:**\n\n1.  **The Hamburger:** The subject of the joke is a hamburger.\n2.  **"A lot of beef":** The joke uses the figurative meaning of "beef" to suggest the hamburger has a lot of unresolved problems or negative experiences from its past.\n3.  **"With its past":** This phrase directly links the "beef" to the 

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({"topic": "pizza"}, config=config2)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to therapy?\n\nBecause it had a lot of **crust** issues!',
 'explanation': 'This joke is a **pun** that plays on the double meaning of the word "**crust**".\n\nHere\'s the breakdown:\n\n*   **Literal Meaning (Pizza):** A pizza has a **crust**, which is the outer edge. This is the part of the pizza that is typically baked and can be hard or sometimes a bit tough.\n\n*   **Figurative Meaning (Therapy):** In the context of therapy, "**crust issues**" is a playful and informal way of referring to **psychological or emotional problems, insecurities, or hang-ups**. It suggests someone has deep-seated issues that are difficult to get to the bottom of, much like it can be hard to get through a thick pizza crust.\n\n**The Joke:**\n\nThe humor comes from the unexpected connection between the literal "crust" of a pizza and the figurative "crust issues" that someone might have in therapy. The pizza, as a personified entity, is seeking help for prob

In [15]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": '1f148ddf-ce30-6193-8000-ea0669a238f6' }})

StateSnapshot(values={'topic': 'burger'}, next=('gen_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f148ddf-ce30-6193-8000-ea0669a238f6'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-05-05T23:56:09.290382+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f148ddf-ce28-6b68-bfff-57a79f4a1174'}}, tasks=(PregelTask(id='1cbed41a-7a08-3e27-fb1c-8bf28c550441', name='gen_joke', path=('__pregel_pull', 'gen_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the hamburger go to therapy?\n\nBecause it had a lot of *beef* with its past!'}),), interrupts=())

In [16]:
workflow.invoke(None, {"thread_id": "1", "checkpoint_id": '1f148ddf-ce30-6193-8000-ea0669a238f6' })

{'topic': 'burger',
 'joke': 'Why did the burger break up with the hot dog?\n\nBecause he felt they were just going through a *bun*-ch of phases!',
 'explanation': 'This joke is a play on words, relying on a pun. Here\'s the breakdown:\n\n*   **"Burger" and "Hot Dog"**: These are common food items, often associated with casual meals and sometimes even parties.\n*   **"Break up"**: This refers to the end of a romantic relationship.\n*   **"Bun"**: This is the bread roll that typically holds a burger.\n*   **"Bunch of phases"**: This is a common idiom meaning someone is going through different stages or periods of change in their life, often implying inconsistency or uncertainty.\n\n**The Pun:**\n\nThe humor comes from the word "**bun**" sounding very similar to "**bunch**".\n\n*   **Literal meaning:** A burger is served in a **bun**.\n*   **Figurative meaning (the pun):** The burger felt like their relationship was going through a **bunch** (many) of changing **phases**, making it unsta

In [18]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger break up with the hot dog?\n\nBecause he felt they were just going through a *bun*-ch of phases!', 'explanation': 'This joke is a play on words, relying on a pun. Here\'s the breakdown:\n\n*   **"Burger" and "Hot Dog"**: These are common food items, often associated with casual meals and sometimes even parties.\n*   **"Break up"**: This refers to the end of a romantic relationship.\n*   **"Bun"**: This is the bread roll that typically holds a burger.\n*   **"Bunch of phases"**: This is a common idiom meaning someone is going through different stages or periods of change in their life, often implying inconsistency or uncertainty.\n\n**The Pun:**\n\nThe humor comes from the word "**bun**" sounding very similar to "**bunch**".\n\n*   **Literal meaning:** A burger is served in a **bun**.\n*   **Figurative meaning (the pun):** The burger felt like their relationship was going through a **bunch** (many) of changing **phase

In [22]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": '1f148ddf-ce30-6193-8000-ea0669a238f6' }, 'checkpoint_ns': ""}, {"topic": "pasta"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f148dfc-92b8-683e-8001-c688212616a5'}}

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pasta'}, next=('gen_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f148dfc-92b8-683e-8001-c688212616a5'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-05-06T00:09:01.517625+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f148ddf-ce30-6193-8000-ea0669a238f6'}}, tasks=(PregelTask(id='dc84e3e7-c435-6ee8-32a2-9ffa6dc37cef', name='gen_joke', path=('__pregel_pull', 'gen_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger break up with the hot dog?\n\nBecause he felt they were just going through a *bun*-ch of phases!', 'explanation': 'This joke is a play on words, relying on a pun. Here\'s the breakdown:\n\n*   **"Burger" and "Hot Dog"**: These are common food items, often associated with casual meals and sometimes even parties.\n*   **"Break

In [24]:
workflow.invoke(None,{"thread_id": "1", "checkpoint_id": '1f148dfc-92b8-683e-8001-c688212616a5' })

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the fettuccine?\n\nBecause he felt like she was always a little too **saucy** for his taste!',
 'explanation': 'This joke is a play on words, using the double meaning of the word "saucy."\n\nHere\'s the breakdown:\n\n*   **Literal Meaning (Food):** Spaghetti and fettuccine are both types of pasta. They are often served with sauce. So, the idea of spaghetti and fettuccine being in a relationship and having a breakup relates to the food they are.\n\n*   **Figurative Meaning (Saucy):**\n    *   **"Saucy" in the context of relationships:** When you say someone is "saucy," it usually means they are **bold, impudent, cheeky, or a bit disrespectful in a playful way.** They might talk back, tease a lot, or have a sharp wit.\n    *   **"Saucy" in the context of food:** This refers to the **sauce** that is put on the pasta.\n\n**The Joke\'s Punchline:**\n\n"Because he felt like she was always a little too **saucy** for his taste!"\

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti break up with the fettuccine?\n\nBecause he felt like she was always a little too **saucy** for his taste!', 'explanation': 'This joke is a play on words, using the double meaning of the word "saucy."\n\nHere\'s the breakdown:\n\n*   **Literal Meaning (Food):** Spaghetti and fettuccine are both types of pasta. They are often served with sauce. So, the idea of spaghetti and fettuccine being in a relationship and having a breakup relates to the food they are.\n\n*   **Figurative Meaning (Saucy):**\n    *   **"Saucy" in the context of relationships:** When you say someone is "saucy," it usually means they are **bold, impudent, cheeky, or a bit disrespectful in a playful way.** They might talk back, tease a lot, or have a sharp wit.\n    *   **"Saucy" in the context of food:** This refers to the **sauce** that is put on the pasta.\n\n**The Joke\'s Punchline:**\n\n"Because he felt like she was always a little too **sauc